<a href="https://colab.research.google.com/github/ManabBiswas/LLM-Agent/blob/main/LLM_AGENT_FOR_DATA_MINING.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install faiss-cpu numpy yfinance datasets sentence-transformers langchain-google-genai scikit-learn langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 3.6 MB/s eta 0:00:00


# Importing dependeceries

In [ ]:
import os
import re
import time
import faiss
import numpy as np
import yfinance as yf
import getpass
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq
from sklearn.metrics import accuracy_score, classification_report

# 1. SECURE API SETUP

In [ ]:
print("="*60)
print("SECURE API INITIALIZATION")
print("="*60)
if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google Gemini API Key: ")
if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")


# 2. HETEROGENEOUS MODEL CONFIGURATION

In [ ]:
print("\n[System] Initializing Deterministic-AI Hybrid Pipeline...")

# Groq handles all high-volume loop calls (Generous free tier for rapid inference)
baseline_llm = ChatGroq(model_name="llama-3.1-8b-instant", temperature=0.1)
pipeline_llm = ChatGroq(model_name="llama-3.3-70b-versatile", temperature=0.1)

# Gemini reserved for ONE final executive synthesis per run (~1 req total)
executive_llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.2)


# 3. DATASET MINING & RAG SETUP

In [ ]:
print("[System] Loading and Indexing Unstructured Dataset...")
# We use this dataset to actually MINE historical sentiment patterns, not just read text.
dataset = load_dataset("zeroshot/twitter-financial-news-sentiment", split="train[:500]")
historical_data = dataset['text']
true_labels = dataset['label'] # 0=Bearish, 1=Bullish, 2=Neutral

embedder = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = embedder.encode(historical_data)
index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(np.array(embeddings))

# 4. DETERMINISTIC TOOLS & MINING LOGIC

In [ ]:
def extract_and_validate_ticker(text):
    """Robust extraction: Looks for $TICKER or known corporate entities."""
    match = re.search(r'\$([A-Z]{1,5})\b', text)
    if match:
        return match.group(1)

    # Fallback dictionary for common mentions without '$'
    known_entities = {"APPLE": "AAPL", "TESLA": "TSLA", "NVIDIA": "NVDA", "MICROSOFT": "MSFT", "AMAZON": "AMZN", "ALPHABET": "GOOGL", "GOOGLE": "GOOGL"}
    for name, ticker in known_entities.items():
        if name in text.upper():
            return ticker
    return None

def mine_historical_sentiment(query_vec, k=5):
    """
    DATA MINING COMPONENT:
    Retrieves K nearest neighbors and applies a deterministic sentiment clustering algorithm.
    Uses whole-word matching to reduce false positives.
    """
    _, idx = index.search(np.array(query_vec), k)
    retrieved_docs = [historical_data[int(i)] for i in idx[0]]

    # Improved: whole-word matching to reduce false positives
    bullish_keywords = [r'\bbuy\b', r'\bsurge\b', r'\bbeat\b', r'\bgrowth\b', r'\brecord\b', r'\bup\b', r'\bhigh\b']
    bearish_keywords = [r'\bsell\b', r'\bdrop\b', r'\bmiss\b', r'\bcut\b', r'\bdowngrade\b', r'\bdown\b', r'\blow\b']

    b = sum(1 for doc in retrieved_docs
            if any(re.search(w, doc.lower()) for w in bullish_keywords))
    bear = sum(1 for doc in retrieved_docs
               if any(re.search(w, doc.lower()) for w in bearish_keywords))

    sentiment = "Bullish" if b > bear else "Bearish" if bear > b else "Neutral"
    return " | ".join(retrieved_docs), sentiment, b, bear

def validate_sentiment_mining(hist_data, labels):
    """Validates the keyword-based sentiment mining against the dataset's true labels."""
    print("\n[VALIDATION] Validating Sentiment Mining Algorithm against True Labels...")
    your_predictions = []

    # Using the same regex logic as mine_historical_sentiment
    bullish_keywords = [r'\bbuy\b', r'\bsurge\b', r'\bbeat\b', r'\bgrowth\b', r'\brecord\b', r'\bup\b', r'\bhigh\b']
    bearish_keywords = [r'\bsell\b', r'\bdrop\b', r'\bmiss\b', r'\bcut\b', r'\bdowngrade\b', r'\bdown\b', r'\blow\b']

    for doc in hist_data:
        b = sum(1 for w in bullish_keywords if re.search(w, doc.lower()))
        bear = sum(1 for w in bearish_keywords if re.search(w, doc.lower()))
        pred = 1 if b > bear else 0 if bear > b else 2
        your_predictions.append(pred)

    print(classification_report(labels, your_predictions, target_names=['Bearish','Bullish','Neutral']))

def fetch_live_price(ticker):
    """Fetches Ground Truth data."""
    try:
        stock = yf.Ticker(ticker)
        data = stock.history(period="1d")
        if not data.empty:
            return round(data['Close'].iloc[-1], 2)
    except:
        pass
    return None

def deterministic_math_verification(draft, true_price):
    """
    NOVELTY: Replaces flawed LLM verification.
    Uses O(1) Python logic to guarantee the AI did not hallucinate the live price.
    Robustly handles formatting differences (e.g., "$150" vs "150.0").
    """
    if true_price is None:
        return True # Nothing to verify mathematically

    # Extract complete decimal numbers first
    numbers_in_draft = set(re.findall(r'\d+\.?\d*', draft))

    # Check within 1% tolerance for rounding edge cases
    for num_str in numbers_in_draft:
        try:
            if abs(float(num_str) - true_price) / true_price < 0.01:
                return True
        except ValueError:
            continue
    return False


# 5. STREAM PROCESSING PIPELINE

In [3]:
def process_data_stream(stream_batch):
    """Processes unstructured data at high speed, collecting metrics for the DWDM report."""
    print("\n" + "="*60)
    print("INITIATING MILLISECOND-SCALE STREAM PROCESSING")
    print("="*60)

    metrics = {"total": len(stream_batch), "processed": 0, "dropped": 0}

    # Separate latency tracking for fair benchmarking
    dropped_latencies = []
    processed_latencies = []
    baseline_latencies = []

    hallucination_count = 0

    for i, text in enumerate(stream_batch):
        triage_start_time = time.time()
        print(f"\n[Stream ID: {i+1}] Ingesting: '{text[:60]}...'")

        # 1. High-Speed Heuristic Triage (Sub-millisecond)
        ticker = extract_and_validate_ticker(text)
        if not ticker:
            print("  -> [Triage] No financial entity detected. Dropping to save latency.")
            metrics["dropped"] += 1
            dropped_latencies.append(time.time() - triage_start_time)
            continue

        print(f"  -> [Entity Extracted]: {ticker}")

        # 2. Ground Truth Fetch
        live_price = fetch_live_price(ticker)
        if not live_price:
            print(f"  -> [API] Live data unavailable for {ticker}. Dropping.")
            metrics["dropped"] += 1
            dropped_latencies.append(time.time() - triage_start_time)
            continue

        # --- FAIR BASELINE COMPARISON (Using Llama-3.1 8B) ---
        base_start = time.time()
        try:
            _ = baseline_llm.invoke(f"Write a 1-sentence analytical brief about: {text}").content
            baseline_latencies.append(time.time() - base_start)
        except Exception as e:
            print(f"  -> [Baseline LLM Error]: {e}")

        time.sleep(1) # Brief pause to prevent rate-limiting between baseline and pipeline

        # --- FULL HYBRID PIPELINE ---
        pipeline_start = time.time()

        # 3. Data Mining (Sentiment Clustering & RAG)
        query_vec = embedder.encode([text])
        context, sentiment, b_score, bear_score = mine_historical_sentiment(query_vec)
        print(f"  -> [Data Mining] Historical Pattern: {sentiment} (Bullish: {b_score}, Bearish: {bear_score})")

        # 4. Generative Synthesis (Using Llama-3.3 70B)
        prompt = f"""
        Task: Write a 1-sentence analytical brief.
        Historical Context (retrieved tweets): {context[:400]}
        Input News: {text}
        Mined Historical Sentiment: {sentiment}
        LIVE PRICE REQUIREMENT: You MUST include the exact current price of ${live_price} in your text. Do not invent other prices.
        """

        try:
            draft = pipeline_llm.invoke(prompt).content.strip()
        except Exception as e:
            print(f"  -> [Pipeline LLM Error]: {e}")
            continue

        # 5. Deterministic Verification (O(1) Time Complexity)
        is_valid = deterministic_math_verification(draft, live_price)

        if not is_valid:
            hallucination_count += 1
            print("  -> [Verification] [FAILED] Math Hallucination Detected! Triggering deterministic fallback.")
            # Deterministic override
            final_output = f"[SYSTEM OVERRIDE] {ticker} sentiment is {sentiment}. Live price verified at ${live_price}."
        else:
            print("  -> [Verification] [PASSED] Deterministic Math Check Passed.")
            final_output = draft

        # Track only the time taken for the actual RAG + LLM + Verification pipeline
        processed_latencies.append(time.time() - pipeline_start)
        metrics["processed"] += 1

        print(f"  -> [Final Insight]: {final_output}")

    # Calculate Benchmark Averages
    baseline_avg = np.mean(baseline_latencies) if baseline_latencies else 0.0
    pipeline_avg = np.mean(processed_latencies) if processed_latencies else 0.0

    # Print Final Benchmarks
    print("\n" + "="*60)
    print("PIPELINE BENCHMARK METRICS")
    print(f"Total Stream Items : {metrics['total']}")
    print(f"Items Processed    : {metrics['processed']}")
    print(f"Items Dropped      : {metrics['dropped']}")

    avg_drop_latency = np.mean(dropped_latencies) if dropped_latencies else 0
    print(f"Avg Triage/Drop Latency          : {avg_drop_latency:.4f}s (Saved API Calls)")

    if baseline_latencies and processed_latencies:
        overhead = ((pipeline_avg - baseline_avg) / baseline_avg) * 100

        print(f"\nBaseline (LLM-only) Avg Latency  : {baseline_avg:.3f}s")
        print(f"Hybrid Pipeline Avg Latency      : {pipeline_avg:.3f}s")
        label = "Overhead" if overhead >= 0 else "Saving"
        print(f"{label} vs Baseline               : {abs(overhead):.1f}%")

    hallucination_rate = (hallucination_count / max(metrics['processed'], 1)) * 100
    print(f"Hallucinations Caught            : {hallucination_count}/{metrics['processed']}")
    print(f"Hallucination Rate (This System) : {hallucination_rate:.1f}%")
    print("="*60)

    # === GEMINI EXECUTIVE SYNTHESIS  ===
    if metrics["processed"] > 0:
        print("\n[Executive Summary] Generating high-level synthesis via Gemini 2.5 Flash...")
        summary_prompt = f"""
        You are a financial data analyst. Based on the following pipeline run statistics,
        write a 5-sentence executive summary of market sentiment observed.

        Items Processed: {metrics['processed']}
        Items Dropped (Noise): {metrics['dropped']}
        Hallucinations Caught: {hallucination_count}
        Hallucination Rate This System: {hallucination_rate:.1f}%
        Baseline Avg Latency: {baseline_avg:.3f}s
        Pipeline Avg Latency: {pipeline_avg:.3f}s
        """
        try:
            summary = executive_llm.invoke(summary_prompt).content.strip()
            print(f"\n>>> EXECUTIVE SUMMARY (Gemini 2.5 Flash):\n{summary}\n")
        except Exception as e:
            print(f"  -> [Executive LLM Error]: {e}")


SECURE API INITIALIZATION

[System] Initializing Deterministic-AI Hybrid Pipeline...
[System] Loading and Indexing Unstructured Dataset...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



[VALIDATION] Validating Sentiment Mining Algorithm against True Labels...
              precision    recall  f1-score   support

     Bearish       0.86      0.31      0.46       153
     Bullish       0.73      0.21      0.33       243
     Neutral       0.24      0.88      0.38       104

    accuracy                           0.38       500
   macro avg       0.61      0.47      0.39       500
weighted avg       0.67      0.38      0.38       500


INITIATING MILLISECOND-SCALE STREAM PROCESSING

[Stream ID: 1] Ingesting: 'I'm going to the mall today....'
  -> [Triage] No financial entity detected. Dropping to save latency.

[Stream ID: 2] Ingesting: 'The weather is nice today in New York....'
  -> [Triage] No financial entity detected. Dropping to save latency.

[Stream ID: 3] Ingesting: 'Just finished reading a great book....'
  -> [Triage] No financial entity detected. Dropping to save latency.

[Stream ID: 4] Ingesting: 'What's for lunch? Pizza or pasta?...'
  -> [Triage] No fin


# 6. EXECUTION


In [ ]:
if __name__ == "__main__":
    # Validate the data mining algorithm first (passing variables explicitly)
    validate_sentiment_mining(historical_data, true_labels)

    # Simulating a high-velocity stream of messy, unstructured data with a statistically significant sample
    simulated_stream = [
        # Noise (6 items)
        "I'm going to the mall today.",
        "The weather is nice today in New York.",
        "Just finished reading a great book.",
        "What's for lunch? Pizza or pasta?",
        "Traffic is terrible this morning.",
        "My cat knocked over my coffee.",

        # Valid Financial (12 items — mix of bullish/bearish)
        "Rumors that Apple might delay their new launch. $AAPL could suffer.",
        "$AAPL iPhone sales beat expectations in China, analysts raise targets.",
        "Nvidia drops new AI chip. $NVDA is looking incredibly strong.",
        "$NVDA faces export restrictions, stock under pressure.",
        "Tesla Q3 deliveries miss expectations, worried about $TSLA margins.",
        "$TSLA Cybertruck demand surges, record pre-orders reported.",
        "$MSFT Azure cloud revenue up 29%, smashing forecasts.",
        "Microsoft faces antitrust probe in EU over $MSFT Teams bundling.",
        "$AMZN Prime Day breaks all records, best day in company history.",
        "Amazon warehouse workers strike spreads to 5 cities. $AMZN faces delays.",
        "$GOOGL ad revenue recovery stronger than expected this quarter.",
        "$GOOGL faces lawsuit over search monopoly, stock retreats."
    ]

    process_data_stream(simulated_stream)